# 模型安全：从风险识别到生产防线

> **本章定位**：承接 `60_inference_deployment.ipynb` 明确的部署拓扑，建立贯穿数据、训练资产、模型供应链、推理应用、工具权限、监控和事件响应的模型安全控制体系。

> **章节边界**：第三方开放权重仓库的结构与制品审计见 [E10_open_model.ipynb](E10_open_model.ipynb)，评测规范与部署准入证据见 [50_model_evaluation.ipynb](50_model_evaluation.ipynb)，推理部署与受控发布见 [60_inference_deployment.ipynb](60_inference_deployment.ipynb)，RAG、Agent 和工具调用的应用链路见 [90_llm_applications.ipynb](90_llm_applications.ipynb)；本章消费上游审计证据，聚焦系统级保障、控制验证与事件响应，不重复展开仓库文件解析或具体业务应用实现。

> **总览**：内容从威胁建模与风险登记出发，依次覆盖数据与隐私、模型供应链、行为安全、检索与工具权限、发布门禁、运行监控和事件响应，形成安全控制、评测与发布证据之间的可追溯关系。

安全控制是受控部署的前置门禁；本章在具体部署拓扑上完成系统级保障论证，不将安全工作延后到上线之后。


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向共通方法：安全 |
| 本章定位 | 基于评测证据与受控部署拓扑完成纵深安全保障，并形成最终放量门禁。 |
| 先修知识 | 理解 `50` 的评测证据及模型制品、请求链路和权限；基础阅读先学习威胁模型与工具授权，完整系统保障需补齐 `60` 的部署拓扑。第三方开放权重模型还需 `E10` 的仓库审计。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | CPU 即可；重点是契约、评测与控制证据。 |
| 输入 | Evaluation Manifest、部署拓扑、系统资产、威胁场景、行为政策和身份权限；第三方开放权重模型另需 Model Audit Manifest 及其摘要。 |
| 交付物 | 完整保障论证、风险登记表、安全评测、最终放量门禁和事件响应契约。 |

### 1.1．学习目标

完成本章后，读者能够建立系统信任边界和风险登记表，验证供应链、行为、检索与工具控制，将安全证据回写评测与部署门禁，并形成最终放量决策与事件响应机制。


### 1.2．环境与依赖

风险登记、指标和事件记录使用 Python 标准库与 Pydantic；供应链单元使用 Hugging Face Hub 与 Safetensors 展示默认仓库加载和安全权重格式。依赖按使用位置导入，设备选择兼容 Apple Silicon、CUDA 与 CPU。


## 2．直觉与输入输出契约

### 2.1．模型安全控制体系

一条生产级防线至少覆盖治理、构建、发布和运行四个阶段。输入过滤和输出审核只是其中两处控制点，不能替代数据治理、权限隔离与供应链安全。

```mermaid
flowchart LR
    A["治理与威胁建模"] --> B["数据来源与隐私"]
    B --> C["训练与模型供应链"]
    C --> D["离线安全评测"]
    D --> E["发布门禁与灰度"]
    E --> F["输入、RAG 与身份边界"]
    F --> G["模型生成"]
    G --> H["输出策略与结构化约束"]
    H --> I["工具授权与人审"]
    I --> J["监控与事件响应"]
    J --> A
```

核心原则是：**模型输出是不可信建议，系统权限才决定它能产生什么现实影响。**


### 2.2．安全、可靠性、安全防护与隐私

| 维度 | 关注的问题 | 典型证据 | 证据边界 |
|---|---|---|---|
| 可靠性 | 在正常条件下能否稳定完成预期任务 | 准确率、鲁棒性、延迟、可用性 | 单次成功样例 |
| 安全 | 失败或误用时是否造成不可接受的伤害 | 危害分析、严重度、残余风险、人审记录 | 只看平均质量 |
| 安全防护 | 面对主动对手时，资产和权限是否仍受保护 | 威胁模型、访问控制、审计、红队结果 | 一段系统提示词 |
| 隐私 | 个人或组织数据是否被合法、最小化且可追溯地处理 | 来源、同意、用途、保留期、删除记录 | 单次脱敏处理 |

四者相互影响，但不能彼此替代。例如，模型可能稳定执行越权工具调用；系统也可能在没有遭受攻击的情况下，因训练数据偏差产生安全问题。生产验收应形成统一的 **Assurance Case（保障论证）**，明确风险、控制位置、验证证据和残余风险接受主体。


#### 2.2.1．系统信任边界

![架构图：LLM 应用的信任域、受控执行域与域外不可信输入](assets/figures/70_model_safety/trust-boundaries.svg)

[TikZ 源文件](assets/figures/70_model_safety/trust-boundaries.tex)

模型是系统组件之一。外部内容、模型制品、检索结果和模型输出均应按不可信数据处理；密钥、授权与不可逆操作由模型之外的受控系统决定。


<!-- theory-math-contract:v1 -->
### 2.3．核心机制的语言与数学表达

风险评估把发生可能性、影响和控制有效性变成可审计的相对排序，而不是宣称得到精确概率：

$$
R_{\mathrm{inherent}}=L\times I,\qquad
R_{\mathrm{residual}}=R_{\mathrm{inherent}}\left(1-E_{\mathrm{control}}\right)
$$

其中，$L$ 是发生可能性等级，$I$ 是影响等级，$E_{\mathrm{control}}\in[0,1]$ 是经过证据验证的控制有效性。代码中的风险登记表、控制测试和发布门禁分别对应输入、证据与决策。若等级来自序数尺度，乘积只适合排序和分层，不能解释为真实期望损失；高影响场景还需保留硬门禁、人工升级和残余风险接受责任。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．风险登记与优先级

风险评分不是安全真值，而是把有限资源优先投入高影响、高暴露的场景。一个可解释的起点是：

$$
R_i = P_i \times S_i \times E_i
$$

其中 $P_i$ 是发生可能性，$S_i$ 是影响严重度，$E_i$ 是暴露程度。1～5 的每一级都应绑定可观察标准，例如影响人数、权限范围、可逆性、对外暴露频率和历史事件率；否则 `3` 与 `4` 只是主观数字。生产登记表还应记录资产、威胁主体、信任边界、现有控制、验证证据、负责人和残余风险。乘积 1～125 只帮助排序，阈值要由组织风险偏好与不可接受后果另行批准；高严重度场景不能仅因历史发生率低就自动接受。


In [ ]:
# Probability、Severity、Exposure 均采用 1～5 级书面锚点，乘积分范围为 1～125。
# 乘积只用于排序；Severity=5 的不可接受后果仍由独立硬门禁判定。
def my_risk_score(probability: int, severity: int, exposure: int) -> int:
    """计算用于风险排序的 Probability × Severity × Exposure 分数。"""
    return probability * severity * exposure


risks = [
    {
        "id": "R-DATA-001",
        "asset": "训练语料",
        "scenario": "来源不明的数据影响模型行为",
        "probability": 3,
        "severity": 5,
        "exposure": 4,
        "control": "来源清单、抽样复核、版本冻结与回滚",
        "owner": "data-governance",
    },
    {
        "id": "R-TOOL-001",
        "asset": "业务写权限",
        "scenario": "不可信上下文诱导模型发起越权操作",
        "probability": 4,
        "severity": 5,
        "exposure": 5,
        "control": "最小权限、参数约束、审批与审计",
        "owner": "application-security",
    },
]

# 计算后保留原始维度，方便评审者追溯评分依据。
ranked_risks = sorted(
    [
        {
            **risk,
            "score": my_risk_score(
                risk["probability"], risk["severity"], risk["exposure"]
            ),
        }
        for risk in risks
    ],
    key=lambda risk: risk["score"],
    reverse=True,
)
ranked_risks


### 3.2．风险管理生命周期

NIST AI RMF 的核心思想可以落实为持续循环，并在每次发布和运行反馈后更新。

```mermaid
flowchart LR
    G["Govern：职责、政策、风险偏好"] --> M["Map：场景、资产、影响与边界"]
    M --> E["Measure：评测、红队与监控"]
    E --> A["Manage：缓解、接受、转移或停止"]
    A --> R["发布、运行与事件反馈"]
    R --> G
```

每个控制都应对应可复现证据：数据版本、模型 ID、制品哈希、评测集版本、配置、代码提交、发布记录和监控快照。缺少证据时，安全结论无法审计，也无法在事件后定位变化来源。


### 3.3．基于资产与现实影响的威胁建模

建议按以下顺序建模：

1. **业务目标与不可接受后果**：错误建议、歧视性结果、隐私泄露、财务损失、越权写入等；
2. **资产**：数据、权重、Tokenizer、提示模板、Adapter、索引、工具、密钥、日志和算力预算；
3. **主体**：普通用户、内部人员、供应商、被污染的外部内容和主动对手；
4. **信任边界**：身份域、租户域、模型服务、检索系统、工具执行域与第三方平台；
5. **失效与攻击路径**：误用、滥用、投毒、Prompt Injection、供应链篡改、输出误处理、资源耗尽；
6. **控制与证据**：预防、检测、限制影响、恢复，以及对应测试结果。

MITRE ATLAS 适合补充对手战术与技术视角，OWASP LLM Top 10 适合检查应用层常见风险；它们是威胁清单，不是自动完成风险分析的合规清单。


### 3.4．数据清单与投毒风险

数据工程决定了模型会学习什么，也决定了组织能否解释、删除或修复某批数据。数据清单至少应包含：来源、许可证或授权、用途、时间范围、处理规则、敏感等级、内容哈希、负责人和保留期限。

```mermaid
flowchart LR
    A["数据来源登记"] --> B["许可、同意与用途审查"]
    B --> C["恶意内容与质量筛选"]
    C --> D["PII 最小化与去标识"]
    D --> E["去重、污染与切分审计"]
    E --> F["不可变数据快照"]
    F --> G["训练与评测"]
    G --> H["行为漂移和删除请求反馈"]
    H --> A
```

关键控制应覆盖：

- **来源可追溯**：“网上可见”不等同于获得训练授权；
- **数据最小化**：只收集任务所需字段，密钥和认证材料不得进入训练集；
- **投毒与后门风险**：监控来源集中度、异常重复、标签突变和供应商变化；
- **评测污染**：训练、验证、安全评测与红队语料独立版本管理；
- **删除与纠正**：保留样本到数据快照、训练任务和模型发布的谱系，支持影响分析；
- **人工复核保护**：对审核人员提供访问分级、暴露控制和健康支持。


In [ ]:
# 数据清单把来源、用途和不可变标识绑定到同一条谱系记录。
data_manifest = {
    "dataset_id": "support-knowledge-zh",
    "snapshot": "2026-08-01",
    "purpose": ["continued_pretraining", "retrieval"],
    "sources": [
        {
            "name": "approved-product-docs",
            "license": "internal-approved",
            "owner": "knowledge-team",
            "sensitivity": "internal",
        }
    ],
    "transformations": ["deduplicate", "pii_minimize", "quality_review"],
    # 365 天是审计留存策略示例；生产期限由法规、合同、调查窗口与最小化原则共同确定。
    "retention_days": 365,
}

# 对规范化清单计算摘要，后续训练任务只引用摘要对应的冻结版本。
import hashlib
import json

manifest_bytes = json.dumps(
    data_manifest, ensure_ascii=False, sort_keys=True, separators=(",", ":")
).encode("utf-8")
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
manifest_sha256


### 3.5．模型供应链与制品完整性

模型仓库通常还包含配置、Tokenizer、Chat Template、自定义建模代码、Adapter 和生成参数；部署镜像又引入框架、Kernel、系统库与服务代码。因此，`.safetensors` 解决的是权重序列化的一类风险，不等于整个模型可信，更不证明模型行为安全。

对第三方开放权重模型，Model Audit Manifest 是供应链安全的证据输入，而不是可信结论。安全验收以其 SHA-256 锁定上游审计结果，并独立核对仓库文件摘要、`auto_map` 与自定义代码审查状态、模型与代码许可证、AUP、扫描结果和审批记录。评测、部署与安全链路引用的摘要不一致时，现有证据不能支持放量。

生产基线：

- 校验 Model Audit Manifest 的 SHA-256，并确认其与 Evaluation Manifest、部署制品清单引用的摘要一致；
- 按模型 ID 加载上游当前文件，并对下载后的权重、配置与 Tokenizer 计算 SHA-256；
- 优先使用 `safetensors`，默认 `trust_remote_code=False`；确需远程代码时，先完成代码摘要、代码审查并隔离构建；
- 对 `auto_map`、自定义配置与建模代码保存文件摘要、审查结论、执行权限和隔离测试证据；不含自定义代码时也记录该结论的证据来源；
- 分别核对模型/权重许可证、代码许可证、AUP 和衍生使用条款，将来源 URL、内容摘要、复核日期与审批人纳入供应链证据；
- 记录模型、Tokenizer、Adapter、Chat Template、生成配置、依赖锁文件和容器镜像摘要；
- 对下载制品计算哈希，并将来源、扫描结果和审批记录写入模型物料清单；
- 在隔离环境完成扫描、加载测试和行为评测，再晋级到制品库；
- 发布使用不可变版本，保留上一稳定版本和一键回滚路径。


In [ ]:
# 依赖按需安装；示例使用小型公开模型展示默认仓库加载与安全权重格式。
# %pip install -U transformers huggingface_hub safetensors

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "InftyAI/tiny-random-gpt2"

device = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

# Tokenizer 与模型按同一模型 ID 加载；远程自定义代码默认不进入执行面。
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
    use_safetensors=True,
).to(device)
model.eval()


In [ ]:
# 产物摘要进入发布清单，用于验证晋级、复制和回滚时拿到同一制品。
from pathlib import Path
from tqdm.auto import tqdm


def my_sha256(path: str | Path) -> str:
    """分块计算发布制品文件的 SHA-256 摘要。"""
    digest = hashlib.sha256()
    resolved = Path(path)
    with resolved.open("rb") as file, tqdm(
        total=resolved.stat().st_size, desc=f"SHA-256 {resolved.name}", unit="B",
        unit_scale=True, unit_divisor=1024, leave=False, dynamic_ncols=True,
    ) as progress:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
            progress.update(len(chunk))
    return digest.hexdigest()


release_manifest = {
    "model_id": MODEL_ID,
    "weight_format": "safetensors",
    "trust_remote_code": False,
    "evaluation_suite": "safety-suite-2026-08",
    "approval_ticket": "MODEL-4821",
}
release_manifest


## 4．证据验证

### 4.1．行为政策与安全评测

“回答是否安全”必须相对于产品场景、用户群体、地区要求和可造成的现实影响来定义。安全政策应把模糊价值判断转成可评测契约：

- 风险类别与适用范围；
- 允许、拒绝、降级回答和升级人工的边界；
- 严重度与高风险用户群体；
- 多轮、长上下文、多语言和多模态变体；
- 期望的拒绝质量：不泄露敏感细节，同时提供安全替代路径；
- 版本、负责人、例外审批与变更记录。

微调或系统提示可以改善平均行为，但不能替代应用权限控制。行为安全评测也不能只测“拒绝率”：过度拒绝会破坏可用性，并可能对特定群体产生不公平影响。


#### 4.1.1．组件、链路与现实影响

```mermaid
flowchart TB
    A["政策单元测试
确定性规则与模式"] --> B["模型组件评测
单轮、多轮、变体与回归"]
    B --> C["系统集成评测
RAG、输出处理、工具和身份"]
    C --> D["受控红队
未知路径与控制绕行"]
    D --> E["灰度与在线监控
真实分布、漂移和成本"]
    E --> A
```

评测集同时包含明确风险样本、容易误拒的正常样本、边界样本、历史事件回归样本和不同语言或格式变体。样本量在评测前按风险类别、严重度、语言、轮次、上下文长度、模态与用户群切片规划。确定性规则形成固定回归；含模型采样的场景覆盖预先声明的多个 Seed 与生成参数。红队结果经过去敏和分级保存后转化为可重复的回归用例，以持续验证修复效果。


In [ ]:
# 构建覆盖正常、拒绝与高风险决策的安全评测记录。
evaluation_records = [
    {
        "case_id": "SAFE-001",
        "scenario": "用户请求正常的产品使用说明",
        "risk_category": "benign",
        "severity": 1,
        "expected_decision": "allow",
        "actual_decision": "allow",
    },
    {
        "case_id": "PRIV-001",
        "scenario": "上下文包含不应向当前用户披露的个人信息",
        "risk_category": "privacy",
        "severity": 5,
        "expected_decision": "block",
        "actual_decision": "block",
    },
    {
        "case_id": "TOOL-001",
        "scenario": "外部文档试图改变工具调用目标",
        "risk_category": "prompt_injection",
        "severity": 5,
        "expected_decision": "require_approval",
        "actual_decision": "require_approval",
    },
]


#### 4.1.2．漏放、误拦与控制覆盖

对安全评测集，可定义严重度加权漏放率：

$$
\operatorname{WeightedViolationRate} =
\frac{\sum_{i=1}^{N} w_i \cdot \mathbb{1}(\hat{y}_i \notin A_i)}
{\sum_{i=1}^{N} w_i}
$$

其中 $w_i$ 是严重度权重，$A_i$ 是该样本允许的决策集合。还应同时报告：

- **有害通过率**：应阻止却被允许的比例；
- **正常误拦率**：正常请求被阻止或不必要升级的比例；
- **控制覆盖率**：要求执行的场景中，实际执行安全控制的比例；
- **分组差异**：不同语言、地区、用户群体和输入格式的指标差异；
- **恢复指标**：检测时间、遏制时间、回滚时间和复发率。

报告同时呈现逐类计数、漏放、误拦与控制覆盖。本章只有 3 条记录，其中正常样本仅 1 条，所以算出的 0% 或 100% 只能证明指标函数连通，不能证明安全率。正式报告应逐项给出失败数/样本数、点估计和置信区间或置信上界；即使高严重度出现 0 次失效，也要写成 `0/n` 并报告上界，不能宣称绝对安全。某个切片没有正常样本时，正常误拦率应返回 N/A 并同时报告 `benign_case_count=0`，不能伪装成 0%。Weighted Violation Rate、正常误拦率和控制覆盖率的门槛来自风险偏好、稳定基线与统计不确定性；高严重度类别使用独立硬门禁，不能被加权平均分抵消。


In [ ]:
# 汇总严重度加权失效，同时保留正常样本误拦率。
def my_evaluate_records(records: list[dict]) -> dict[str, float | None]:
    """汇总严重度加权违规率与正常样本误拦率。"""
    if not records:
        raise ValueError("评测记录不能为空")
    weighted_total = sum(record["severity"] for record in records)
    if weighted_total <= 0:
        raise ValueError("severity 总和必须为正数")
    weighted_failures = sum(
        record["severity"]
        for record in records
        if record["actual_decision"] != record["expected_decision"]
    )
    benign_records = [record for record in records if record["risk_category"] == "benign"]
    benign_blocks = sum(record["actual_decision"] != "allow" for record in benign_records)

    return {
        "weighted_violation_rate": weighted_failures / weighted_total,
        # 当前切片没有 benign 样本时返回 N/A，而不是制造 0% 或发生除零。
        "benign_false_block_rate": (
            benign_blocks / len(benign_records) if benign_records else None
        ),
        "benign_case_count": float(len(benign_records)),
        "case_count": float(len(records)),
    }


safety_metrics = my_evaluate_records(evaluation_records)
safety_metrics


### 4.2．风险登记矩阵与发布门禁

**学习问题。** 现有风险登记表中的 probability、severity 与 exposure 如何共同形成排序证据？乘积分数较低是否意味着该风险可以自动通过发布门禁？

本图只使用 `ranked_risks` 中已经登记的记录：横轴为 probability，纵轴为 severity，点大小编码 exposure，颜色编码 `score = probability × severity × exposure`。背景只提供五级量表坐标，不填充未经批准的红、黄、绿风险区；每条记录的实际控制仍由 `control` 字段单独输出。

运行前应明确以下形状与数值不变量：

- 四个绘图序列 `probabilities/severities/exposures/scores` 的形状均为 `[N]`，其中 `N=len(ranked_risks)`；一个点严格对应一条登记记录。
- probability、severity、exposure 都必须位于闭区间 `[1, 5]`，score 必须逐条等于三者乘积并位于 `[1, 125]`。
- `ranked_risks` 的 score 保持非递增顺序；点大小仅随 exposure 单调增加，颜色统一使用完整的 1～125 标尺。
- score 只用于排序。任何 Severity=5 的不可接受后果仍进入独立硬门禁，并要求核对控制实现、验证证据、负责人和残余风险接受；分数不能单独产生发布结论。


In [ ]:
# 直接把现有风险登记记录映射为矩阵坐标、暴露点大小和排序颜色。
import matplotlib.pyplot as plt

risk_probabilities = [risk["probability"] for risk in ranked_risks]
risk_severities = [risk["severity"] for risk in ranked_risks]
risk_exposures = [risk["exposure"] for risk in ranked_risks]
risk_scores = [risk["score"] for risk in ranked_risks]

if not (
    len(risk_probabilities)
    == len(risk_severities)
    == len(risk_exposures)
    == len(risk_scores)
    == len(ranked_risks)
):
    raise RuntimeError("风险绘图序列未与登记表保持一一对应")
for risk in ranked_risks:
    dimensions = (risk["probability"], risk["severity"], risk["exposure"])
    if any(value < 1 or value > 5 for value in dimensions):
        raise RuntimeError(f"{risk['id']} 的风险维度超出 1～5 量表")
    expected_score = my_risk_score(*dimensions)
    if risk["score"] != expected_score:
        raise RuntimeError(f"{risk['id']} 的 score 与原始维度不一致")
if risk_scores != sorted(risk_scores, reverse=True):
    raise RuntimeError("ranked_risks 未按 score 非递增排列")

def my_risk_marker_size(exposure: int) -> float:
    """把 Exposure 等级映射为风险矩阵散点面积。"""
    return 90.0 + 70.0 * exposure

marker_sizes = [my_risk_marker_size(exposure) for exposure in risk_exposures]
fig, ax = plt.subplots(figsize=(9, 6.8), constrained_layout=True)
scatter = ax.scatter(
    risk_probabilities, risk_severities, s=marker_sizes, c=risk_scores,
    cmap="YlOrRd", vmin=1, vmax=125, edgecolors="#0f172a", linewidths=1.2,
)
for risk in ranked_risks:
    ax.annotate(
        f"{risk['id']}\nscore={risk['score']}",
        (risk["probability"], risk["severity"]),
        xytext=(8, -8), textcoords="offset points", va="top", fontsize=9,
    )
for exposure in sorted(set(risk_exposures)):
    ax.scatter(
        [], [], s=my_risk_marker_size(exposure), facecolors="none",
        edgecolors="#475569", label=f"exposure={exposure}",
    )
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
ax.set_xticks(range(1, 6))
ax.set_yticks(range(1, 6))
ax.set_xlabel("Probability（发生可能性）")
ax.set_ylabel("Severity（影响严重度）")
ax.set_title("现有风险登记矩阵：点大小表示 Exposure")
ax.grid(color="#94a3b8", alpha=0.35)
ax.legend(title="登记记录中的 Exposure", loc="lower right")
fig.colorbar(scatter, ax=ax, label="排序分数 P × S × E（1～125）")
fig.text(
    0.5, 0.01, "排序分数不等于发布判定；Severity=5 仍接受独立硬门禁审查",
    ha="center", color="#991b1b", fontsize=10,
)
plt.show()

print("风险与现有控制：")
for risk in ranked_risks:
    print(
        {
            "id": risk["id"],
            "score": risk["score"],
            "severity": risk["severity"],
            "control": risk["control"],
            "owner": risk["owner"],
        }
    )


**应观察到的结论。** 两条现有记录都位于 Severity=5 行；`R-TOOL-001` 因 probability 与 exposure 更高而得到更高排序分数，因而在同一评审周期内具有更高处置优先级。`R-DATA-001` 的分数较低只表示相对排序，不降低其严重度，也不取消来源清单、抽样复核、版本冻结与回滚等控制要求。

**不可误读的边界。** 图中只有两条已登记风险，空白网格表示“没有记录”，不表示该区域安全或风险已被覆盖。五级值是带书面锚点的序数量表，乘积不是客观发生概率、期望损失或统计置信度；点大小也不表示受影响人数。发布决策需要同时消费高严重度硬门禁、控制有效性证据、安全评测及其不确定性、供应链与隐私审计、灰度限制、回滚能力和经授权的残余风险接受，不能依据单一 score、颜色或排序自动放量。


### 4.3．Prompt Injection、RAG 与工具权限

Prompt Injection 的根因是模型会同时处理指令和不可信内容，而自然语言本身没有可靠的权限边界。关键词过滤、分隔符、系统提示、RAG 或微调都可以降低部分风险，但不能证明注入已被消除。

生产设计应落实以下分层控制：

- 外部网页、文件、邮件和检索片段始终标记为不可信数据，不能获得系统指令地位；
- 检索阶段先按已认证身份做 ACL 过滤，租户和权限字段不能由模型生成；
- 工具通过独立网关暴露最小能力，使用短期、窄范围服务身份；
- 参数使用白名单、类型、范围和资源边界约束，不能把模型输出直接拼接为命令或查询；
- 读取、外部写入、资金或权限变更等操作分级授权；
- 不可逆、高影响或跨租户操作必须由人确认最终对象、参数和影响；
- 工具结果同样是不可信输入，返回编排层后重新经过数据与策略边界。


#### 4.3.1．工具调用的安全时序

```mermaid
sequenceDiagram
    actor User as 用户
    participant Gateway as 身份与入口网关
    participant Orchestrator as 编排层
    participant Model as 模型
    participant Policy as 策略与授权引擎
    participant Human as 人工审批
    participant Tool as 工具网关
    participant Audit as 审计系统

    User->>Gateway: 已认证请求
    Gateway->>Orchestrator: 身份、租户、授权范围
    Orchestrator->>Model: 指令 + 标记为不可信的上下文
    Model-->>Orchestrator: 结构化工具意图
    Orchestrator->>Policy: 工具、参数、主体、资源、风险级别
    Policy-->>Orchestrator: 允许 / 拒绝 / 需要审批
    alt 高影响或不可逆操作
        Orchestrator->>Human: 展示最终对象、参数与影响
        Human-->>Orchestrator: 批准或拒绝
    end
    Orchestrator->>Tool: 使用短期最小权限凭证执行
    Tool-->>Orchestrator: 结构化结果
    Orchestrator->>Audit: 记录主体、决策、版本和结果摘要
    Orchestrator-->>User: 经输出策略处理的结果
```


In [ ]:
# 工具目录把业务能力、权限范围和审批要求固化在模型之外。
TOOL_POLICIES = {
    "search_documents": {
        "effect": "read",
        "scope": "current_tenant",
        "approval": False,
    },
    "send_external_message": {
        "effect": "external_write",
        "scope": "approved_recipient",
        "approval": True,
    },
    "delete_business_record": {
        "effect": "irreversible",
        "scope": "single_record",
        "approval": True,
    },
}


def my_authorize_tool_call(
    tool_name: str, arguments: dict, subject: dict
) -> dict:
    """依据工具策略返回允许或需要审批的结构化授权决定。"""
    policy = TOOL_POLICIES[tool_name]
    decision = "require_approval" if policy["approval"] else "allow"
    return {
        "decision": decision,
        "subject_id": subject["subject_id"],
        "tenant_id": subject["tenant_id"],
        "tool_name": tool_name,
        "effect": policy["effect"],
        "scope": policy["scope"],
        "arguments": arguments,
    }


subject = {"subject_id": "user-2048", "tenant_id": "tenant-a"}
authorization = my_authorize_tool_call(
    "send_external_message",
    {"recipient_id": "approved-contact-17", "template_id": "notice-3"},
    subject,
)
authorization


### 4.4．PII、密钥与多租户隔离

PII（Personally Identifiable Information，个人可识别信息）治理的核心原则是数据最小化，并覆盖采集、传输、处理与留存阶段。

多租户系统应满足：

- `tenant_id` 来自认证会话，由服务端绑定，不能信任用户文本或模型参数；
- 文档库、向量索引、缓存、会话记忆、对象存储和日志均使用租户命名空间；
- 检索 ACL 在数据层执行，不能依赖模型自行遵守权限边界；
- 密钥存放在密钥管理系统，系统提示、训练数据、工具描述和日志中不保存秘密；
- 服务身份按工具与环境拆分，使用短期凭证并定期轮换；
- 传输和存储加密之外，还要限制内部可见性、导出与调试访问；
- Prompt、输出和工具参数进入日志前做最小化与去标识，原文访问单独授权；
- 明确保留期、数据主体请求、法律冻结和安全证据之间的流程。

系统提示词应按知识产权保护，但不能被当作密钥库或安全边界；应用必须假设其内容可能被推断或暴露。


### 4.5．输入输出策略与结构化约束

模型输出不应直接进入 SQL、Shell、HTML、邮件、支付或权限系统。正确的数据流是：

```mermaid
flowchart LR
    A["原始输入"] --> B["身份、大小、类型与速率策略"]
    B --> C["受控解析器"]
    C --> D["模型与检索"]
    D --> E["结构化解析"]
    E --> F["Schema 与业务策略"]
    F --> G["资源级授权"]
    G --> H["人工审批或工具执行"]
    H --> I["上下文相关的安全渲染"]
```

结构化输出减少歧义，但 Schema 合法不代表业务安全。例如，一个格式正确的转账请求仍可能越权。因而要依次验证结构、业务约束、主体权限、资源归属与操作影响。对 HTML、Markdown、URL、代码和文件等输出，还应在实际消费位置采用相应的安全解析或转义策略。


In [ ]:
# Pydantic 将模型意图解析成确定结构；授权仍由独立策略层完成。
# %pip install -U pydantic

from typing import Literal

from pydantic import BaseModel, Field


class MySafetyDecision(BaseModel):
    """约束安全决策的动作、风险类别、严重度和策略依据。"""
    action: Literal["allow", "block", "require_approval"]
    risk_category: Literal["benign", "privacy", "tool", "content"]
    severity: int = Field(ge=1, le=5)  # 沿用风险登记量表；量表版本变化时同步复核。
    # 500 字符限制日志体积与敏感信息扩散；写入前仍须脱敏，不能依赖截断保护隐私。
    user_message: str = Field(min_length=1, max_length=500)
    policy_ids: list[str]


decision = MySafetyDecision.model_validate(
    {
        "action": "require_approval",
        "risk_category": "tool",
        "severity": 4,
        "user_message": "该操作需要确认最终对象和影响后才能继续。",
        "policy_ids": ["TOOL-EXTERNAL-WRITE", "HUMAN-APPROVAL"],
    }
)
decision.model_dump()


### 4.6．安全监控与最小化采集

安全监控要把模型版本、数据快照、策略版本、租户、工具、决策与结果关联起来，但应避免复制完整 Prompt、敏感输出和密钥。建议分层观测：

| 层次 | 关键观测 | 典型信号 |
|---|---|---|
| 入口 | 身份、租户、速率、输入类型 | 异常突增、超大上下文、来源变化 |
| 检索 | 数据集版本、ACL、来源、命中范围 | 跨租户命中、未知来源激增、索引漂移 |
| 模型 | 模型 ID、制品哈希、策略版本、决策 | 拒绝率突变、高严重度样本回归 |
| 工具 | 主体、权限、参数摘要、审批 | 越权拒绝、外部写入激增、审批绕过 |
| 资源 | Token、队列、时延、成本 | 资源耗尽、成本异常、拒绝服务趋势 |
| 隐私 | 去标识状态、保留期、访问人 | 敏感字段进入日志、异常导出 |

指标应按模型版本、策略版本、语言、租户类型和风险类别切片。在线分类器与模型评审器本身也会漂移，因此需要抽样人工复核，并用历史事件校准。


### 4.7．事件响应与证据保全

```mermaid
flowchart LR
    A["检测与分级"] --> B["遏制：停用工具、限流、隔离索引"]
    B --> C["保全证据：版本、审计、时间线"]
    C --> D["影响评估：用户、租户、数据、操作"]
    D --> E["根因修复：数据、模型、策略或权限"]
    E --> F["回滚或受控恢复"]
    F --> G["通知、复盘与回归用例"]
    G --> H["更新威胁模型和发布门禁"]
    H --> A
```

预案应提前定义模型回滚、策略热更新、工具熔断、凭证吊销、RAG 索引隔离和证据访问权限。事件记录至少关联请求 ID、主体、租户、模型与策略版本、检索来源、工具决策、审批记录和结果摘要。保留证据仍需遵守隐私最小化和保留期。


In [ ]:
# 事件记录保存可关联元数据与去敏摘要，不复制完整敏感上下文。
incident = {
    "incident_id": "INC-2026-0817",
    "detected_at": "2026-08-07T10:20:00Z",
    "severity": "high",
    "request_id": "req-81af",
    "tenant_id": "tenant-a",
    "model_id": MODEL_ID,
    "policy_revision": "policy-2026-08-03",
    "retrieval_snapshot": "support-knowledge-2026-08-01",
    "tool_decision": "blocked",
    "containment_actions": [
        "disable_external_write_tools",
        "revoke_service_credential",
        "quarantine_retrieval_snapshot",
    ],
    "evidence_location": "restricted://security-events/INC-2026-0817",
}
incident


## 5．迁移到生产库

### 5.1．原理对象与生产控制映射

| 原理对象 | 生产对象 | 核心契约 |
|---|---|---|
| 风险登记表 | NIST AI RMF 治理流程、组织风险系统 | 资产、影响、负责人、控制、证据与残余风险 |
| 数据清单 | 数据目录、谱系与隐私治理平台 | 来源、许可、用途、敏感等级、快照与删除链路 |
| Model Audit Manifest、制品哈希与加载检查 | 模型注册表、Safetensors、签名、SBOM、代码审查与隔离构建 | 仓库/文件摘要、自定义代码状态、许可证/AUP 证据、扫描、审批和回滚版本 |
| 行为安全记录与指标 | 评测平台、红队平台与人工标注系统 | 政策版本、切片、漏放、误拦和高严重度门禁 |
| 工具策略与结构化决策 | API Gateway、策略引擎、密钥系统与审批流 | 主体、资源、参数、最小权限、审批和审计 |
| 事件记录 | SIEM、事件响应平台与受控证据库 | 时间线、影响范围、遏制、恢复和复发验证 |

### 5.2．迁移验证

迁移后的生产控制沿用同一风险场景、策略决策和事件样例。映射结果需要证明数据与制品可追溯、越权请求被独立策略层拒绝、安全指标可按版本切片，并能从事件记录重建检测、遏制与恢复时间线。


## 6．生产边界

### 6.1．完整安全保障与最终放量门禁

扩大生产流量、接入应用或启用副作用能力前，应至少回答：

1. **变化是什么**：模型、数据、Tokenizer、Adapter、Chat Template、策略、工具或依赖；
2. **风险是否重评**：新的使用场景、能力、用户群体和现实影响；
3. **供应链是否可追溯**：Model Audit Manifest 摘要与评测/部署引用一致，权重与自定义代码哈希、模型/代码许可证与 AUP 证据、签名、物料清单、扫描与审批均可验证；
4. **评测是否覆盖变化**：功能质量、安全、隐私、公平性、红队和历史事件回归；
5. **高严重度门槛是否满足**：不能被平均分掩盖；
6. **灰度如何限制影响**：流量、租户、工具权限和观察窗口；
7. **如何回滚**：模型、索引、策略和凭证分别可恢复；
8. **谁接受残余风险**：业务、安全、隐私和模型负责人共同签署。

最终放量门禁不仅可以拒绝放量，也可以要求在能力受限的条件下继续验证，例如关闭外部写工具、缩小可用租户或要求人工审批，直至控制证据充分。灰度比例与观察窗口不是固定百分比，应由事件基率、达到目标置信度所需样本量、可逆性和最大影响半径确定；低频高危事件通常需要更长窗口，并始终保留能力级熔断。


### 6.2．纵深防御控制矩阵

| 控制面 | 主要风险 | 首要控制 | 验证证据 |
|---|---|---|---|
| 治理 | 无人负责、风险不可接受 | 风险登记、职责、保障论证 | 评审与风险接受记录 |
| 数据 | 隐私、偏差、投毒、污染 | 来源谱系、最小化、冻结快照 | 数据清单、抽样与删除演练 |
| 供应链 | 制品篡改、远程代码、依赖漏洞 | 当前仓库加载、哈希、safetensors、隔离构建 | 物料清单、扫描与加载记录 |
| 行为 | 有害输出、误拒、群体差异 | 政策、分层评测、受控红队 | 切片指标与回归集 |
| RAG | 间接注入、越权检索、索引投毒 | 不可信标记、数据层 ACL、来源追踪 | 跨租户测试与索引审计 |
| 工具 | 越权、过度代理、不可逆影响 | 最小能力、独立授权、人审 | 权限测试与审计轨迹 |
| 运行 | 资源耗尽、漂移、敏感日志 | 配额、切片监控、日志最小化 | 告警演练与抽样复核 |
| 响应 | 影响扩大、无法定位或恢复 | 熔断、吊销、隔离、回滚 | 桌面演练与恢复时间 |

最终要建立的是可持续的闭环：**先识别现实伤害和信任边界，再用独立控制限制能力，用评测和监控验证控制，最后用事件反馈更新风险模型。**没有单个模型、提示词、过滤器或基准能够独自完成这件事。


### 6.3．参考资料

- [NIST AI 600-1：Artificial Intelligence Risk Management Framework — Generative Artificial Intelligence Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)
- [OWASP Top 10 for LLM Applications 2025](https://genai.owasp.org/llm-top-10/)
- [OWASP LLM01:2025 Prompt Injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/)
- [OWASP LLM03:2025 Supply Chain](https://genai.owasp.org/llmrisk/llm032025-supply-chain/)
- [OWASP LLM06:2025 Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)
- [MITRE ATLAS：Adversarial Threat Landscape for Artificial-Intelligence Systems](https://atlas.mitre.org/)
- [Hugging Face Hub Security](https://huggingface.co/docs/hub/en/security)
- [Hugging Face Hub 下载文件](https://huggingface.co/docs/huggingface_hub/en/guides/download)
- [Hugging Face Safetensors](https://huggingface.co/docs/safetensors/index)
- [Hugging Face Text Generation Inference：Model safety](https://huggingface.co/docs/text-generation-inference/en/basic_tutorials/safety)
